# 14 — Blend CatBoost (nous) × Graphe-XGBoost (corrigé)

On a deux modèles PROPRES : notre CatBoost (last 0.3607) et le graphe-XGBoost du notebook 13
(last 0.3593). Question : sont-ils assez décorrélés pour que le blend dépasse 0.3607 ?
On charge `oof_graph.csv` / `test_graph.csv` et on mesure en CV temporelle (boussole LB ≈ last − 0.004).

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from src import config as C
from src.validation import time_folds, evaluate_ap
from src.utils import op03_mask, seed_everything, make_submission
from src.features.temporal import balance_features, recency_features
from src.features.behavioral import behavioral_features
from src.encoding import oof_target_encode_train, fit_target_map, apply_target_map, recent_target_rate
seed_everything(42)
DATA = ROOT / "data"; NB = ROOT / "notebooks"
train = pd.read_csv(DATA / "train.csv"); test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
op03 = op03_mask(train).to_numpy(); y_all = train[C.TARGET].to_numpy()
folds_full = list(time_folds(train[C.PERIOD]))

In [ ]:
EPS = 1e-6; WINDOWS = (5, 10, 20); SMOOTHING = 30
def row_features(df):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]; f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    return pd.concat([f, balance_features(df)], axis=1)
def add_freq(X, src_df, ref_df):
    X = X.copy()
    for col in [C.ORIGIN_ACCT, C.DEST_ACCT]:
        freq = ref_df[col].value_counts(normalize=True)
        X[f"freq_{col}"] = src_df[col].map(freq).fillna(0).values
    return X
def base_build(df, ref):
    X = row_features(df).reset_index(drop=True)
    X = add_freq(X, df.reset_index(drop=True), ref)
    beh = behavioral_features(df, ref).reset_index(drop=True)
    rec = recency_features(df, ref).reset_index(drop=True)
    rt = recent_target_rate(df, ref, C.ORIGIN_ACCT, C.PERIOD, C.TARGET, WINDOWS).reset_index(drop=True)
    return pd.concat([X, beh, rec, rt], axis=1)
def feats_train(df, ref):
    X = base_build(df, ref); X["te_origin"] = oof_target_encode_train(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SMOOTHING); return X
def feats_apply(df, ref):
    X = base_build(df, ref); mp, gm = fit_target_map(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SMOOTHING)
    X["te_origin"] = apply_target_map(df, C.ORIGIN_ACCT, mp, gm); return X
def make_cat():
    from catboost import CatBoostClassifier
    return CatBoostClassifier(loss_function="Logloss", eval_metric="PRAUC", depth=6,
                              learning_rate=0.05, iterations=600, random_seed=42, verbose=False)

## OOF CatBoost + chargement OOF graphe, alignés par id

In [ ]:
# 1) OOF CatBoost (notre champion)
oof_cat = np.zeros(len(train))
for tr_idx, va_idx in folds_full:
    tr_op = tr_idx[op03[tr_idx]]; va_op = va_idx[op03[va_idx]]; ref = train.iloc[tr_op]
    m = make_cat().fit(feats_train(train.iloc[tr_op], ref), y_all[tr_op])
    oof_cat[va_op] = m.predict_proba(feats_apply(train.iloc[va_op], ref))[:, 1]

# 2) OOF graphe (notebook 13) aligné par id
g = pd.read_csv(NB / "oof_graph.csv").set_index("id")["oof"]
oof_graph = train[C.ID].map(g).fillna(0).to_numpy()

# 3) AP par fold : cat, graphe, blend
def rk(x):
    return np.argsort(np.argsort(x)) / (len(x) - 1)
rows = []
for k, (_, va_idx) in enumerate(folds_full):
    vo = va_idx[op03[va_idx]]
    ap_c = evaluate_ap(y_all[vo], oof_cat[vo])
    ap_g = evaluate_ap(y_all[vo], oof_graph[vo])
    ap_b = evaluate_ap(y_all[vo], 0.5 * rk(oof_cat[vo]) + 0.5 * rk(oof_graph[vo]))
    rows.append((ap_c, ap_g, ap_b))
R = np.array(rows)
print("           cat     graphe   blend")
for k, r in enumerate(R):
    print(f"fold {k}:  {r[0]:.4f}  {r[1]:.4f}  {r[2]:.4f}")
print(f"\nlast   :  {R[-1,0]:.4f}  {R[-1,1]:.4f}  {R[-1,2]:.4f}")
print(f"recent2:  {R[-2:,0].mean():.4f}  {R[-2:,1].mean():.4f}  {R[-2:,2].mean():.4f}")
print("corr cat-graphe (op_03) :", round(np.corrcoef(oof_cat[op03], oof_graph[op03])[0, 1], 3))

## Si blend last > 0.3607 : soumission du blend test

In [ ]:
# CatBoost final sur tout le train op_03 -> proba test
ref_full = train.iloc[np.where(op03)[0]]
final = make_cat().fit(feats_train(ref_full, ref_full), y_all[op03])
te_op = op03_mask(test).to_numpy()
cat_test = np.zeros(len(test))
cat_test[te_op] = final.predict_proba(feats_apply(test.iloc[np.where(te_op)[0]], ref_full))[:, 1]

# proba graphe test (notebook 13) alignée par id
gt = pd.read_csv(NB / "test_graph.csv").set_index("id")["target"]
graph_test = test[C.ID].map(gt).fillna(0).to_numpy()

# blend équipondéré par rang, sur op_03 uniquement
full = np.zeros(len(test))
full[te_op] = 0.5 * rk(cat_test[te_op]) + 0.5 * rk(graph_test[te_op])
path = make_submission(test[C.ID], full, "14_blend_cat_graph")
sub = pd.read_csv(path)
assert list(sub.columns) == ["id", "target"] and len(sub) == len(test)
assert set(sub["id"]) == set(sample["id"]) and sub["target"].between(0, 1).all()
print("soumission écrite :", path, "| proba>0 :", int((sub['target'] > 0).sum()))